# 環境情報モデルをTensorRTへの変換

- 30_detect

で作成した学習済みモデルをTensorRTフォーマットに変換します。

In [ ]:
import os

# ---------- 1. Jetson.GPIO 読み取り ----------
try:
    import Jetson.GPIO as GPIO
    BOARD_NAME = GPIO.gpio_pin_data.get_data()[0]
except Exception as e:
    # 失敗したら Orin Nano と決め打ち
    print(f"[WARN] Jetson モデル判定エラー: {e} → 強制的に JETSON_ORIN_NANO として続行")
    os.environ["JETSON_MODEL_NAME"] = "JETSON_ORIN_NANO"
    import Jetson.GPIO as GPIO          # もう一度ロード
    BOARD_NAME = "JETSON_ORIN_NANO"     # 確定

# ---------- 2. ボード別定義 ----------
mode_descriptions = {
    "JETSON_NX":       ["15W_2CORE", "15W_4CORE", "15W_6CORE", "10W_2CORE", "10W_4CORE"],
    "JETSON_XAVIER":   ["MAXN", "MODE_10W", "MODE_15W", "MODE_30W"],
    "JETSON_NANO":     ["MAXN", "5W"],
    "JETSON_ORIN":     ["MAXN", "MODE_15W", "MODE_30W", "MODE_40W"],
    "JETSON_ORIN_NANO":["MODE_15W", "MODE_25W", "MODE_MAX"]
}

product_names = {
    "JETSON_NX":        "Jetson Xavier NX",
    "JETSON_XAVIER":    "Jetson AGX Xavier",
    "JETSON_NANO":      "Jetson Nano",
    "JETSON_ORIN":      "Jetson AGX Orin",
    "JETSON_ORIN_NANO": "Jetson Orin Nano"
}

# (I2C バス番号, 初期 Power モードインデックス)
board_settings = {
    "JETSON_NX":        (8, 3),
    "JETSON_XAVIER":    (8, 2),
    "JETSON_NANO":      (1, 0),
    "JETSON_ORIN":      (7, 0),
    "JETSON_ORIN_NANO": (7, 2)
}

# ---------- 3. パラメータ取得 ----------
i2c_busnum, power_mode = board_settings.get(BOARD_NAME, (None, None))
mode_list       = mode_descriptions.get(BOARD_NAME, [])
product_name    = product_names.get(BOARD_NAME, "未知のボード")

# ---------- 4. 出力 ----------
if i2c_busnum is not None and 0 <= power_mode < len(mode_list):
    mode_str = mode_list[power_mode]
    print("------------------------------------------------------------")
    print(f"{product_name} を認識: I2C バス番号 = {i2c_busnum}, "
          f"Power モード = {mode_str} ({power_mode})")
    print("------------------------------------------------------------")
else:
    raise RuntimeError(f"未対応の Jetson モデル、または Power モード定義不足: {BOARD_NAME}")

In [ ]:
if (product_name == "Jetson Orin Nano") or (product_name == "Jetson AGX Orin"):
    print("Docker起動のため電力モードは変更できません。")
else:
    !echo "jetson" | sudo -S nvpmodel -m $power_mode

In [ ]:
!echo "jetson" | sudo -S nvpmodel -q

In [ ]:
if (product_name == "Jetson Orin Nano") or (product_name == "Jetson AGX Orin"):
    print("Docker起動のためjetson_clocksは起動できません。")
else:
    !echo "jetson" | sudo -S jetson_clocks

In [ ]:
import os

import torch
import torchvision
from packaging import version

from fabo import EnvironmentCategory

CATEGORIES = [x.name for x in EnvironmentCategory]

output_dim = len(CATEGORIES)
device = torch.device('cuda')

torchvision_version = version.parse(torchvision.__version__)

# torchvision 0.13以降の場合
if torchvision_version >= version.parse("0.13"):
    from torchvision.models.resnet import ResNet18_Weights, resnet18

    model = resnet18(weights=None)  # pretrained=Falseの代わり
    model.fc = torch.nn.Linear(model.fc.in_features, output_dim)
else:
    model = torchvision.models.resnet18(pretrained=False)
    model.fc = torch.nn.Linear(512, output_dim)

model = model.cuda().eval().half()

In [ ]:
from torch2trt import torch2trt

In [ ]:
import ipywidgets
from ipywidgets import Button, Layout, Textarea, HBox, VBox, Label
import os
import glob

l = Layout(flex='0 1 auto', height='100px', min_height='100px', width='auto')
process_widget = ipywidgets.Textarea(description='ログ', value='', layout=l)

process_no = 0
def write_log(msg):
    global process_widget, process_no
    process_no = process_no + 1
    process_widget.value = str(process_no) + ": " + msg + "\n" + process_widget.value

In [ ]:
import datetime
import glob
import os
import subprocess
import time
from pathlib import Path

import ipywidgets
from ipywidgets import Button, Layout, Textarea, HBox, VBox, Label

from fabo import asset_root

load_model_widget = ipywidgets.Dropdown(options=[],description='変換前')
load_model_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px"))
load_model_time_widget = ipywidgets.Label(description='作成日時：')

trt_model_widget = ipywidgets.Text(description='変換後', value="result.pth")
convert_button = ipywidgets.Button(description='TRT Convert')


def load_model_list(change):
    global load_model_widget
    try:
        files = glob.glob(os.path.join(asset_root(), 'model_class', '*.pth'), recursive=True)
        load_model_widget.options = files
        if len(load_model_widget.options) > 0 and load_model_widget.value not in load_model_widget.options:
            load_model_widget.index = 0

        ts = os.path.getctime(files[0])
        d = datetime.datetime.fromtimestamp(ts)
        s = d.strftime('%Y-%m-%d %H:%M:%S')
        load_model_time_widget.value = f'作成日時：{s}'
    except:
        load_model_widget.options = []
load_model_refresh_button.on_click(load_model_list)
load_model_list(None)

def change_file(change):
    file = load_model_widget.value
    ts = os.path.getctime(file)
    d = datetime.datetime.fromtimestamp(ts)
    s = d.strftime('%Y-%m-%d %H:%M:%S')
    load_model_time_widget.value = f'作成日時：{s}'
load_model_widget.observe(change_file, names='value')


def convert_trt(change):
    global model, model_trt
    try:
        write_log("モデルの変換を開始します。")
        start_time = time.time()
        model_path = Path(load_model_widget.value)
        write_log(f"変換前: {model_path}")
        try:                # PyTorch ≥ 2.1
            state_dict = torch.load(model_path, weights_only=True)
        except TypeError:   # PyTorch 2.0 系など fallback
            state_dict = torch.load(model_path, map_location='cpu')
        model.load_state_dict(state_dict)
        write_log("TRTへの変換処理中...(時間がかかります)")
        data = torch.zeros((1, 3, 224, 224)).cuda().half()
        model_trt = torch2trt(model, [data], fp16_mode=True)
        path = os.path.join(asset_root(), "model_class_trt")
        os.makedirs(path, exist_ok=True)
        trt_model_path = os.path.join(path, trt_model_widget.value)
        torch.save(model_trt.state_dict(), trt_model_path)
        write_log("保存成功")
        end_time = time.time() - start_time
        write_log(f"---------------------------------")
        write_log(f"処理時間　　　: {round(end_time,2)}秒")
        write_log(f"変換前のモデル: {model_path}")
        write_log(f"変換後のモデル: {trt_model_path}")
        write_log(f"---------------------------------")
    except Exception as e:
        write_log(f"Error:{e}")
    
convert_button.on_click(convert_trt)

In [ ]:
import glob
import re

import cv2
import torch.nn.functional as F

from fabo import asset_root
from utils import preprocess

IMG_WIDTH = 224
IMG_HEIGHT = 224
LOAD_TASK = ['camera','train','run','interactive','cropData']
SKIP = [1,2,3,4,5]

movie_button = ipywidgets.Button(description='動画の作成')
movie_name_widget = ipywidgets.Text(description='動画名',value="run_video_detect_trt")
load_datasets_dropdown = ipywidgets.Dropdown(options=[], description='dataset')
load_datasets_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px"))
load_task_dropdown = ipywidgets.Dropdown(options=LOAD_TASK, description='task')
movie_skip_dropdown = ipywidgets.Dropdown(options=SKIP, description='skip(枚)', index=1)

def change_load_task(change):
    path = os.path.join(asset_root(), load_task_dropdown.value)
    try:
        files = os.listdir(path)
        dirs = [f for f in files if os.path.isdir(os.path.join(path, f))]
        dirs = [f for f in files if f != ".ipynb_checkpoints"]
        dirs = sorted(dirs)
        load_datasets_dropdown.options = dirs
        if len(load_datasets_dropdown.options) > 0 and load_datasets_dropdown.value not in load_datasets_dropdown.options:
            load_datasets_dropdown.index = 0
    except:
        write_log(path + "が存在していません。")
        load_datasets_dropdown.options = []
load_datasets_refresh_button.on_click(change_load_task)
load_task_dropdown.observe(change_load_task, names='value')
change_load_task(None)

def extract_numbers(filename):
    matches = re.findall(r'(\d+)', filename)
    if matches and len(matches) >= 3: 
        return int(matches[-1])  
    else:
        return float('inf')

def make_movie(change):
    global model_trt
    
    if not movie_name_widget.value.strip():
        write_log("ファイル名を指定してください。")
        return 
    write_log("動画を作成します。")
    path = os.path.join(asset_root(), "video")
    os.makedirs(path, exist_ok=True)
    output = os.path.join(path, movie_name_widget.value + ".mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = int(30/movie_skip_dropdown.value)
    outfh = cv2.VideoWriter(output, fourcc, fps, (224, 224))

    file_list = sorted(
        glob.glob(os.path.join(asset_root(), load_task_dropdown.value, load_datasets_dropdown.value, 'xy', '*.jpg')),
        key=lambda f: extract_numbers(os.path.basename(f)),
    )

    try:
        res_num = len(file_list)
        skip_movie = movie_skip_dropdown.value
        terminal_time = 1 / (30 / skip_movie)
        current_time = 0
        process_time = 0
        total_process_time = 0
        for i, file_name in enumerate(file_list):
            if i % skip_movie == 0:
                current_time += terminal_time
                img = cv2.imread(file_name)

                process_time = time.time()
                preprocessed = preprocess(img).half()
                output = model_trt(preprocessed)
                output = F.softmax(output, dim=1).detach().cpu().numpy().flatten()
                category_index = output.argmax()
                # write_log(f"Predicted category: {category_index}")

                thickness = 2  # テキストの太さ
                font_scale = 0.5  # フォントの大きさ
                total_process_time += time.time() - process_time

                img = cv2.putText(img, f"C={category_index}", (10, 200), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 255), thickness)
                img = cv2.putText(img, f"I={i}", (60, 200), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 255), thickness)
        
                for h, score in enumerate(list(output)):
                    if h == category_index:
                        img = cv2.putText(img, f"{CATEGORIES[h]} : {int(score * 100)} %", (10, h * 15 + 40), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 255), thickness)
                    else:
                        img = cv2.putText(img, f"{CATEGORIES[h]} : {int(score * 100)} %", (10, h * 15 + 40), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness)

                if i%(skip_movie*10) == 0:
                    write_log(f"{current_time:.1f}秒まで完了, 推論平均: {total_process_time/10*1000:.1f}ms, {int(i/skip_movie)}枚目/{int(res_num/skip_movie)}枚中を処理中")
                    total_process_time = 0
                outfh.write(img)
                time.sleep(5/1000)
                del img
    except Exception as e:
        write_log(f"Error:{e}")
    finally:
        # エラーが発生しても確実にリソースを解放する
        outfh.release()
        write_log("動画の出力が完了しました。")
        #get_jetson_nano_memory_usage()

movie_button.on_click(make_movie)

In [ ]:
separator = ipywidgets.HTML('<hr style="border-color:gray;margin:10px 0"/>')
title1 = ipywidgets.HTML('<b>【1.TensorRTへの変換】</b> 軽量化、高速化を目的にモデルをTensorRT形式に変換します。')
title2 = ipywidgets.HTML('<b>【2.変換後のモデルの動画評価】</b> 変換後のモデルで動画を作成し評価します。')

convert_widget = ipywidgets.VBox([
    separator,
    title1,
    ipywidgets.HBox([load_model_widget, load_model_refresh_button, load_model_time_widget]),
    ipywidgets.HBox([trt_model_widget]),
    convert_button,
    process_widget,
    separator,
    title2,
    ipywidgets.HBox([load_datasets_dropdown, load_datasets_refresh_button, load_task_dropdown]),
    ipywidgets.HBox([movie_name_widget,movie_skip_dropdown,movie_button]),
    process_widget,
])
display(convert_widget)